# 탐지기 명세서 확인 및 데이터 구성

### 1. 단일 클래스 탐지기 입력 준비 — 최종 데이터 연결 확인

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# 1. 이전에 생성한 최종 데이터 경로
DETECTOR_DATA_ROOT = Path(
    "/Users/codeit/Desktop/코드잇/project/pill-project/data"
)

DETECTOR_PACKAGE_DIR = (
    DETECTOR_DATA_ROOT
    / "integrated_dataset_20261027_142006_359958"
    / "split_20261027_143217_217659"
    / "revision_20261027_182411_405580"
    / "training_package"
)

DETECTOR_SELECT_ROOT = DETECTOR_DATA_ROOT / "select"

# 2. 필요한 파일 확인
detector_required_files = {
    "Train 라벨": DETECTOR_PACKAGE_DIR / "annotations" / "train.json",
    "Validation 라벨": (
        DETECTOR_PACKAGE_DIR / "annotations" / "validation.json"
    ),
    "공통 분할 목록": DETECTOR_PACKAGE_DIR / "split_manifest.csv",
}

detector_missing_files = [
    str(path)
    for path in detector_required_files.values()
    if not path.is_file()
]

if detector_missing_files:
    raise FileNotFoundError(
        "필요한 파일을 찾지 못했습니다.\n"
        + "\n".join(detector_missing_files)
    )

if not DETECTOR_SELECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"SELECT 폴더를 찾지 못했습니다: {DETECTOR_SELECT_ROOT}"
    )

# 3. 저장된 최종 라벨 읽기
detector_coco = {}

for split_name in ["train", "validation"]:
    path = (
        DETECTOR_PACKAGE_DIR / "annotations" / f"{split_name}.json"
    )
    with path.open("r", encoding="utf-8") as file:
        detector_coco[split_name] = json.load(file)

detector_manifest = pd.read_csv(
    DETECTOR_PACKAGE_DIR / "split_manifest.csv",
    encoding="utf-8-sig",
)

# 4. 최종 확정 수량과 비교
detector_expected_counts = {
    "train": (8256, 31673),
    "validation": (2063, 7915),
}

detector_count_rows = []

for split_name, labels in detector_coco.items():
    images = labels["images"]
    annotations = labels["annotations"]

    expected_images, expected_boxes = detector_expected_counts[split_name]

    assert len(images) == expected_images, (
        f"{split_name} 이미지 수가 예상과 다릅니다. "
        f"실제 {len(images):,} / 예상 {expected_images:,}"
    )
    assert len(annotations) == expected_boxes, (
        f"{split_name} 박스 수가 예상과 다릅니다. "
        f"실제 {len(annotations):,} / 예상 {expected_boxes:,}"
    )

    detector_count_rows.append({
        "분할": split_name,
        "이미지 수": len(images),
        "박스 수": len(annotations),
        "원본 약품 클래스 수": len({
            annotation["category_id"]
            for annotation in annotations
        }),
    })

# 5. 필수 구조와 이미지 ID 연결 확인
detector_all_image_ids = set()

for split_name, labels in detector_coco.items():
    image_ids = [image["id"] for image in labels["images"]]
    annotation_ids = [
        annotation["id"] for annotation in labels["annotations"]
    ]

    assert len(image_ids) == len(set(image_ids)), (
        f"{split_name}: 이미지 ID 중복"
    )
    assert len(annotation_ids) == len(set(annotation_ids)), (
        f"{split_name}: annotation ID 중복"
    )
    assert all(
        isinstance(image_id, int)
        and not isinstance(image_id, bool)
        and image_id >= 0
        for image_id in image_ids
    ), f"{split_name}: 이미지 ID 형식 오류"

    split_image_ids = set(image_ids)

    assert not detector_all_image_ids.intersection(split_image_ids), (
        "Train과 Validation 사이에 이미지 ID가 중복됩니다."
    )

    assert all(
        annotation["image_id"] in split_image_ids
        for annotation in labels["annotations"]
    ), f"{split_name}: 이미지에 연결되지 않은 annotation이 있습니다."

    detector_all_image_ids.update(split_image_ids)

assert len(detector_manifest) == 10319, (
    f"분할 목록은 10,319행이어야 합니다. "
    f"실제 {len(detector_manifest):,}행"
)

# 6. 다음 단계에서 사용할 실제 저장 구조 출력
print("========== 최종 데이터 연결 확인 ==========")
print(f"학습 패키지: {DETECTOR_PACKAGE_DIR}")
print(f"SELECT 이미지 기준 폴더: {DETECTOR_SELECT_ROOT}")

display(pd.DataFrame(detector_count_rows))

print("\n[분할 목록의 열]")
print(detector_manifest.columns.tolist())

print("\n[분할 목록 첫 번째 기록]")
display(detector_manifest.head(1))

print("\n[Train 이미지 기록 예시]")
print(json.dumps(
    detector_coco["train"]["images"][0],
    ensure_ascii=False,
    indent=2,
))

print("\n[Train annotation 기록 예시]")
print(json.dumps(
    detector_coco["train"]["annotations"][0],
    ensure_ascii=False,
    indent=2,
))

print("\n[통과] 추가 5장 제외 후 최종 이미지·박스 수 확인")
print("[통과] 전체 이미지 ID 중복 없음")
print("[통과] 각 분할의 annotation ID 중복 없음")
print("[통과] annotation과 이미지 ID 연결 확인")
print("\n※ 아직 DataLoader를 생성한 단계는 아닙니다.")
print("※ 파일 수정·삭제 없음")

# 2. Dataset·DataLoader 생성

In [ ]:
# ============================================================
# 2. Dataset·DataLoader 생성
# ============================================================

from pathlib import Path
from types import SimpleNamespace
import importlib.util
import json
import shutil
import sys
import zipfile

try:
    import torch
    import numpy as np
    from PIL import Image
except ImportError as error:
    raise RuntimeError(
        "현재 노트북 환경에 torch, numpy, Pillow가 필요합니다. "
        f"누락된 항목: {error}"
    ) from error


# ------------------------------------------------------------
# 1. 앞 단계에서 확인한 경로
# ------------------------------------------------------------
package_dir = Path(DETECTOR_PACKAGE_DIR)
select_root = Path(DETECTOR_SELECT_ROOT)
data_root = Path(DETECTOR_DATA_ROOT)

codeit_zip = (
    data_root
    / "codeit"
    / "codeit_validated_20261027_132901_622959"
    / "codeit_train_validated.zip"
)

DETECTOR_DELIVERY_DIR = package_dir.parent / "detector_data_delivery"
module_dir = DETECTOR_DELIVERY_DIR / "data"
module_dir.mkdir(parents=True, exist_ok=True)

# 대회 이미지는 필요한 파일만 별도 폴더에 준비합니다.
# SELECT 이미지는 복사하지 않습니다.
codeit_image_root = DETECTOR_DELIVERY_DIR / "codeit_images"

required_codeit_images = set()

for split_name in ["train", "validation"]:
    label_path = package_dir / "annotations" / f"{split_name}.json"

    with label_path.open("r", encoding="utf-8") as file:
        document = json.load(file)

    for image_info in document["images"]:
        portable_path = image_info["file_name"].replace("\\", "/")
        if portable_path.startswith("codeit/"):
            required_codeit_images.add(
                portable_path.split("/", 1)[1]
            )

missing_codeit_images = [
    relative
    for relative in sorted(required_codeit_images)
    if not (codeit_image_root / relative).is_file()
]

if missing_codeit_images:
    if not codeit_zip.is_file():
        raise FileNotFoundError(
            f"대회 이미지 ZIP을 찾지 못했습니다: {codeit_zip}"
        )

    with zipfile.ZipFile(codeit_zip) as archive:
        members = set(archive.namelist())

        for relative in missing_codeit_images:
            relative_path = Path(relative)

            if relative_path.is_absolute() or ".." in relative_path.parts:
                raise ValueError(f"허용되지 않는 이미지 경로: {relative}")

            if relative not in members:
                raise FileNotFoundError(
                    f"대회 ZIP 내부에서 이미지를 찾지 못했습니다: {relative}"
                )

        for relative in missing_codeit_images:
            destination = codeit_image_root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)

            with archive.open(relative) as source:
                with destination.open("wb") as target:
                    shutil.copyfileobj(source, target)


# ------------------------------------------------------------
# 2. 재사용할 데이터 구성 모듈
# ------------------------------------------------------------
detector_module_source = r'''
"""고정 분할을 사용하는 단일 클래스 알약 탐지 데이터 구성."""

from __future__ import annotations

import csv
import hashlib
import json
import math
import random
import re
from collections import defaultdict
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
from PIL import Image, ImageEnhance
from torch.utils.data import DataLoader, Dataset, get_worker_info

if TYPE_CHECKING:
    from detector import DetectorConfig


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


def original_file_name(file_name):
    name = Path(file_name).name
    match = re.search(r"(K-\d{6}.*)$", name)

    if match is None:
        raise ValueError(f"원본 파일명을 확인할 수 없습니다: {name}")

    return match.group(1)


def canonical_group(file_name):
    prefix = original_file_name(file_name).split("_", 1)[0]
    codes = prefix.removeprefix("K-").split("-")

    if not codes or not all(
        re.fullmatch(r"\d{6}", code) for code in codes
    ):
        raise ValueError(f"알약 조합 형식 오류: {file_name}")

    return "K-" + "-".join(sorted(codes))


def collate_detector_samples(samples):
    return {
        "images": [sample["image"] for sample in samples],
        "targets": [sample["target"] for sample in samples],
        "meta": [sample["meta"] for sample in samples],
    }


def seed_detector_worker(worker_id):
    info = get_worker_info()
    if info is not None:
        info.dataset.rng.seed(torch.initial_seed())


class PillDetectionDataset(Dataset):
    def __init__(self, records, augmentation, seed):
        self.records = records
        self.augmentation = dict(augmentation)
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]

        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")

        width, height = image.size

        if (width, height) != (record["width"], record["height"]):
            raise ValueError(
                f"실제 이미지와 JSON 크기 불일치: {record['file_name']}"
            )

        boxes = torch.tensor(
            record["boxes"], dtype=torch.float32
        ).reshape(-1, 4)

        horizontal = self.augmentation["horizontal_flip"]
        vertical = self.augmentation["vertical_flip"]

        if horizontal > 0 and self.rng.random() < horizontal:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            left = boxes[:, 0].clone()
            right = boxes[:, 2].clone()
            boxes[:, 0] = width - right
            boxes[:, 2] = width - left

        if vertical > 0 and self.rng.random() < vertical:
            image = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
            top = boxes[:, 1].clone()
            bottom = boxes[:, 3].clone()
            boxes[:, 1] = height - bottom
            boxes[:, 3] = height - top

        brightness = self.augmentation["brightness"]
        contrast = self.augmentation["contrast"]

        if brightness > 0:
            image = ImageEnhance.Brightness(image).enhance(
                self.rng.uniform(1 - brightness, 1 + brightness)
            )

        if contrast > 0:
            image = ImageEnhance.Contrast(image).enhance(
                self.rng.uniform(1 - contrast, 1 + contrast)
            )

        image_tensor = torch.from_numpy(
            np.array(image, dtype=np.uint8, copy=True)
        ).permute(2, 0, 1).contiguous().to(torch.float32).div_(255)

        boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, width)
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, height)

        if not torch.all(boxes[:, 2:] > boxes[:, :2]):
            raise ValueError(
                f"변환 후 박스 면적 오류: 이미지 {record['image_id']}"
            )

        count = len(boxes)

        target = {
            "boxes": boxes,
            "labels": torch.ones(count, dtype=torch.int64),
            "image_id": record["image_id"],
            "area": (
                (boxes[:, 2] - boxes[:, 0])
                * (boxes[:, 3] - boxes[:, 1])
            ),
            "iscrowd": torch.zeros(count, dtype=torch.int64),
        }

        meta = {
            "image_id": record["image_id"],
            "source_image_id": record["source_image_id"],
            "file_name": record["original_file_name"],
            "image_path": record["image_path"],
            "group_id": record["group_id"],
            "original_size": [height, width],
            "source_annotation_count": count,
            "annotation_ids": list(record["annotation_ids"]),
            "bbox_xywh": [list(box) for box in record["bbox_xywh"]],
            "source_category_ids": list(record["category_ids"]),
            "drug_codes": list(record["drug_codes"]),
            "dataset": record["dataset"],
            "transformed_boxes_xyxy": boxes.tolist(),
        }

        return {
            "image": image_tensor,
            "target": target,
            "meta": meta,
        }


def validate_batch(batch):
    for key in ["images", "targets", "meta"]:
        if not isinstance(batch.get(key), list):
            raise ValueError(f"배치의 {key}는 리스트여야 합니다.")

    if not batch["images"]:
        raise ValueError("빈 배치입니다.")

    if not (
        len(batch["images"])
        == len(batch["targets"])
        == len(batch["meta"])
    ):
        raise ValueError("배치 리스트 길이가 일치하지 않습니다.")

    for image, target, meta in zip(
        batch["images"], batch["targets"], batch["meta"]
    ):
        assert image.device.type == "cpu", "이미지는 CPU에 있어야 합니다."
        assert image.dtype == torch.float32, "이미지 자료형 오류"
        assert image.ndim == 3 and image.shape[0] == 3, "이미지 형태 오류"
        assert image.shape[1] > 0 and image.shape[2] > 0
        assert torch.isfinite(image).all(), "이미지 숫자 오류"
        assert image.min() >= 0 and image.max() <= 1, "이미지 범위 오류"

        height, width = image.shape[-2:]
        boxes = target["boxes"]
        count = len(boxes)

        assert count > 0, "박스 없는 이미지"
        assert boxes.dtype == torch.float32
        assert boxes.shape == (count, 4)
        assert torch.isfinite(boxes).all()
        assert torch.all(boxes[:, 0] >= 0)
        assert torch.all(boxes[:, 1] >= 0)
        assert torch.all(boxes[:, 2] <= width)
        assert torch.all(boxes[:, 3] <= height)
        assert torch.all(boxes[:, 2:] > boxes[:, :2])

        assert target["labels"].dtype == torch.int64
        assert torch.all(target["labels"] == 1)
        assert target["area"].dtype == torch.float32
        assert target["iscrowd"].dtype == torch.int64
        assert torch.all(target["iscrowd"] == 0)

        for field in ["labels", "area", "iscrowd"]:
            assert target[field].shape == (count,)

        expected_area = (
            (boxes[:, 2] - boxes[:, 0])
            * (boxes[:, 3] - boxes[:, 1])
        )
        assert torch.allclose(target["area"], expected_area)

        assert type(target["image_id"]) is int
        assert target["image_id"] >= 0
        assert target["image_id"] == meta["image_id"]

        assert count == meta["source_annotation_count"]
        assert count == len(meta["annotation_ids"])
        assert count == len(meta["bbox_xywh"])
        assert len(set(meta["annotation_ids"])) == count

        assert isinstance(meta["source_image_id"], str)
        assert meta["file_name"] and meta["image_path"] and meta["group_id"]

    return True


def build_detector_inputs(*, config: DetectorConfig, device):
    settings = read_json(
        Path(__file__).with_name("detector_data_settings.json")
    )

    package = Path(settings["package_dir"]).expanduser().resolve()
    roots = {
        name: Path(value).expanduser().resolve()
        for name, value in settings["image_roots"].items()
    }

    with (package / "split_manifest.csv").open(
        "r", encoding="utf-8-sig", newline=""
    ) as file:
        manifest_rows = list(csv.DictReader(file))

    manifest = {}

    for row in manifest_rows:
        image_id = int(row["image_id"])

        if image_id in manifest:
            raise ValueError(f"분할 목록의 이미지 ID 중복: {image_id}")

        manifest[image_id] = row

    all_image_ids = set()
    all_annotation_ids = set()
    records_by_split = {}
    clipped_annotation_ids = []

    for split in ["train", "validation"]:
        labels = read_json(package / "annotations" / f"{split}.json")
        split_ids = set()
        annotation_groups = defaultdict(list)

        for image in labels["images"]:
            image_id = image["id"]

            if (
                type(image_id) is not int
                or image_id < 0
                or image_id in all_image_ids
            ):
                raise ValueError(f"이미지 ID 오류 또는 중복: {image_id}")

            all_image_ids.add(image_id)
            split_ids.add(image_id)

        for annotation in labels["annotations"]:
            annotation_id = annotation["id"]

            if (
                type(annotation_id) is not int
                or annotation_id in all_annotation_ids
            ):
                raise ValueError(
                    f"annotation ID 오류 또는 중복: {annotation_id}"
                )

            if annotation["image_id"] not in split_ids:
                raise ValueError(
                    f"이미지에 연결되지 않은 annotation: {annotation_id}"
                )

            all_annotation_ids.add(annotation_id)
            annotation_groups[annotation["image_id"]].append(annotation)

        records = []

        for image in labels["images"]:
            image_id = image["id"]

            if image_id not in manifest:
                raise ValueError(f"분할 목록에 없는 이미지: {image_id}")

            row = manifest[image_id]

            if row["split"] != split:
                raise ValueError(f"분할 배정 불일치: {image_id}")

            width = int(image["width"])
            height = int(image["height"])

            if width <= 0 or height <= 0:
                raise ValueError(f"이미지 크기 오류: {image_id}")

            portable = str(image["file_name"]).replace("\\", "/")
            dataset, relative = portable.split("/", 1)
            relative_path = Path(relative)

            if relative_path.is_absolute() or ".." in relative_path.parts:
                raise ValueError(f"허용되지 않는 이미지 경로: {portable}")

            if dataset not in roots:
                raise ValueError(f"이미지 기준 폴더 설정 없음: {dataset}")

            if (
                dataset != row["dataset"]
                or portable != row["image_path"]
            ):
                raise ValueError(f"이미지 경로 기록 불일치: {image_id}")

            group = canonical_group(portable)

            if group != row["group_id"]:
                raise ValueError(f"조합 그룹 기록 불일치: {image_id}")

            actual_path = roots[dataset] / relative_path

            if not actual_path.is_file():
                raise FileNotFoundError(actual_path)

            annotations = annotation_groups[image_id]

            if not annotations:
                raise ValueError(f"박스 없는 이미지: {image_id}")

            if len(annotations) != int(row["annotation_count"]):
                raise ValueError(f"annotation 개수 불일치: {image_id}")

            boxes = []
            source_boxes = []

            for annotation in annotations:
                annotation_id = annotation["id"]
                raw_box = annotation["bbox"]

                if len(raw_box) != 4:
                    raise ValueError(f"bbox 원소 수 오류: {annotation_id}")

                x, y, w, h = map(float, raw_box)

                if not all(math.isfinite(v) for v in [x, y, w, h]):
                    raise ValueError(f"bbox 숫자 오류: {annotation_id}")

                if w <= 0 or h <= 0:
                    raise ValueError(f"bbox 크기 오류: {annotation_id}")

                if annotation.get("iscrowd", 0) != 0:
                    raise ValueError(
                        f"iscrowd가 0이 아닙니다: {annotation_id}"
                    )

                if annotation.get("ignore", 0) != 0:
                    raise ValueError(
                        f"ignore가 0이 아닙니다: {annotation_id}"
                    )

                before = [x, y, x + w, y + h]
                box = [
                    min(max(x, 0), width),
                    min(max(y, 0), height),
                    min(max(x + w, 0), width),
                    min(max(y + h, 0), height),
                ]

                if box[2] <= box[0] or box[3] <= box[1]:
                    raise ValueError(
                        f"경계 처리 후 박스 면적이 0입니다: {annotation_id}"
                    )

                if box != before:
                    clipped_annotation_ids.append(annotation_id)

                boxes.append(box)
                source_boxes.append([x, y, w, h])

            records.append({
                "image_id": image_id,
                "source_image_id": str(row["original_image_id"]),
                "original_file_name": original_file_name(portable),
                "portable_path": portable,
                "image_path": str(actual_path.resolve()),
                "dataset": dataset,
                "group_id": group,
                "width": width,
                "height": height,
                "boxes": boxes,
                "bbox_xywh": source_boxes,
                "annotation_ids": [a["id"] for a in annotations],
                "category_ids": [int(a["category_id"]) for a in annotations],
                "drug_codes": [
                    a.get("drug_code", f"K-{int(a['category_id']):06d}")
                    for a in annotations
                ],
            })

        records_by_split[split] = sorted(
            records, key=lambda record: record["image_id"]
        )

    if set(manifest) != all_image_ids:
        raise ValueError("분할 목록과 라벨의 전체 이미지 구성이 다릅니다.")

    groups = {
        split: {record["group_id"] for record in records}
        for split, records in records_by_split.items()
    }

    overlap = groups["train"] & groups["validation"]

    if overlap:
        raise ValueError(
            f"Train과 Validation의 조합 그룹 중복: {len(overlap)}개"
        )

    label_map = {
        "num_foreground_classes": 1,
        "num_model_classes": 2,
        "background_label": 0,
        "classes": [{"label": 1, "name": "pill"}],
    }

    hash_records = [
        {
            "split": split,
            "image_id": record["image_id"],
            "group_id": record["group_id"],
            "file_name": record["portable_path"],
            "annotation_ids": record["annotation_ids"],
            "bbox_xywh": record["bbox_xywh"],
            "category_ids": record["category_ids"],
        }
        for split in ["train", "validation"]
        for record in records_by_split[split]
    ]

    split_hash = hashlib.sha256(
        json.dumps(
            {"version": "fixed-pill-data-v1", "records": hash_records},
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()

    augmentation = settings["augmentation"]
    required_options = {
        "horizontal_flip", "vertical_flip", "brightness", "contrast"
    }

    if set(augmentation) != required_options:
        raise ValueError("증강 설정 항목이 올바르지 않습니다.")

    for name, value in augmentation.items():
        if (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
            or not math.isfinite(value)
        ):
            raise ValueError(f"증강 설정의 숫자 오류: {name}")

        upper = 1.0 if name.endswith("flip") else 0.3

        if not 0 <= value <= upper:
            raise ValueError(f"증강 설정 범위 오류: {name}")

    summary = {
        "group_overlap_count": 0,
        "split_hash": split_hash,
        "clipped_box_count": len(clipped_annotation_ids),
        "clipped_annotation_ids": clipped_annotation_ids,
        "train_augmentation": dict(augmentation),
        "validation_augmentation": "none",
    }

    for split, name in [("train", "train"), ("validation", "val")]:
        records = records_by_split[split]
        counts = [len(record["boxes"]) for record in records]

        if not counts:
            raise ValueError(f"{split}: 이미지가 없습니다.")

        instance_count = sum(counts)

        summary.update({
            f"num_{name}_images": len(records),
            f"num_{name}_source_annotations": instance_count,
            f"num_{name}_instances": instance_count,
            f"{name}_group_count": len(groups[split]),
            f"{name}_empty_image_count": 0,
            f"{name}_max_instances_per_image": max(counts),
            f"{name}_label_counts": {"1": instance_count},
        })

    workers = 2 if device.type == "cuda" else 0
    pin_memory = device.type == "cuda"
    loaders = []

    for split, offset in [("train", 0), ("validation", 1)]:
        transforms = (
            augmentation if split == "train"
            else {name: 0.0 for name in augmentation}
        )

        dataset = PillDetectionDataset(
            records_by_split[split],
            transforms,
            config.seed + offset,
        )

        generator = torch.Generator()
        generator.manual_seed(config.seed + offset)

        loaders.append(DataLoader(
            dataset,
            batch_size=config.batch_size,
            shuffle=split == "train",
            num_workers=workers,
            pin_memory=pin_memory,
            drop_last=False,
            collate_fn=collate_detector_samples,
            worker_init_fn=seed_detector_worker,
            generator=generator,
        ))

    return loaders[0], loaders[1], label_map, summary
'''


# ------------------------------------------------------------
# 3. 모듈과 데이터 경로 설정 저장
# ------------------------------------------------------------
# 문법 검사: 오류가 있으면 파일 저장 전에 중단합니다.
compile(detector_module_source, "detector_data.py", "exec")

# 주석·문자열에 한자 또는 일본어 문자가 포함되면 중단합니다.
unexpected_characters = {
    char
    for char in detector_module_source
    if (
        "\u3040" <= char <= "\u30ff"
        or "\u3400" <= char <= "\u4dbf"
        or "\u4e00" <= char <= "\u9fff"
        or "\uff66" <= char <= "\uff9f"
    )
}

if unexpected_characters:
    raise ValueError("코드에 확인이 필요한 문자가 포함되어 저장을 중단했습니다.")

module_path = module_dir / "detector_data.py"
settings_path = module_dir / "detector_data_settings.json"

settings = {
    "package_dir": str(package_dir),
    "image_roots": {
        "select": str(select_root),
        "codeit": str(codeit_image_root),
    },
    "augmentation": {
        "horizontal_flip": 0.0,
        "vertical_flip": 0.0,
        "brightness": 0.0,
        "contrast": 0.0,
    },
}

# 이전 생성 코드·설정이 있으면 사본을 보관합니다.
for path in [module_path, settings_path]:
    if path.is_file():
        shutil.copy2(path, path.with_name(path.name + ".previous"))

module_path.write_text(detector_module_source, encoding="utf-8")
settings_path.write_text(
    json.dumps(settings, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

init_path = module_dir / "__init__.py"
if not init_path.exists():
    init_path.write_text("", encoding="utf-8")


# ------------------------------------------------------------
# 4. 생성한 모듈을 독립적으로 불러오기
# ------------------------------------------------------------
module_name = "pill_detector_data_check"
spec = importlib.util.spec_from_file_location(module_name, module_path)
detector_data_module = importlib.util.module_from_spec(spec)
sys.modules[module_name] = detector_data_module
spec.loader.exec_module(detector_data_module)

# 팀원의 DetectorConfig를 변경하거나 다시 정의하지 않습니다.
# 이번 독립 검사에서는 데이터 함수가 사용하는 두 항목만 제공합니다.
detector_check_config = SimpleNamespace(seed=42, batch_size=8)

if torch.cuda.is_available():
    detector_check_device = torch.device("cuda")
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    detector_check_device = torch.device("mps")
else:
    detector_check_device = torch.device("cpu")


# ------------------------------------------------------------
# 5. 전체 라벨 구조 검사와 DataLoader 생성
# ------------------------------------------------------------
train_loader, val_loader, detector_label_map, detector_data_summary = (
    detector_data_module.build_detector_inputs(
        config=detector_check_config,
        device=detector_check_device,
    )
)

expected_counts = {
    "num_train_images": 8256,
    "num_val_images": 2063,
    "num_train_instances": 31673,
    "num_val_instances": 7915,
}

for field, expected in expected_counts.items():
    actual = detector_data_summary[field]
    assert actual == expected, (
        f"{field} 불일치: 실제 {actual:,}, 예상 {expected:,}"
    )

train_check_batch = next(iter(train_loader))
val_check_batch = next(iter(val_loader))

detector_data_module.validate_batch(train_check_batch)
detector_data_module.validate_batch(val_check_batch)

# 같은 설정으로 다시 구성했을 때 첫 배치와 집계가 같은지 확인
(
    repeated_train_loader,
    repeated_val_loader,
    repeated_label_map,
    repeated_summary,
) = detector_data_module.build_detector_inputs(
    config=detector_check_config,
    device=detector_check_device,
)

assert repeated_label_map == detector_label_map
assert repeated_summary == detector_data_summary

for loader, previous_batch in [
    (repeated_train_loader, train_check_batch),
    (repeated_val_loader, val_check_batch),
]:
    repeated_batch = next(iter(loader))
    detector_data_module.validate_batch(repeated_batch)

    assert [
        item["image_id"] for item in previous_batch["meta"]
    ] == [
        item["image_id"] for item in repeated_batch["meta"]
    ]

    for before, after in zip(
        previous_batch["images"], repeated_batch["images"]
    ):
        assert torch.equal(before, after)

    for before, after in zip(
        previous_batch["targets"], repeated_batch["targets"]
    ):
        assert torch.equal(before["boxes"], after["boxes"])

# 첫 배치 확인이 끝났으므로 실제 사용할 로더를 다시 생성
train_loader, val_loader, _, _ = (
    detector_data_module.build_detector_inputs(
        config=detector_check_config,
        device=detector_check_device,
    )
)


# ------------------------------------------------------------
# 6. 검사 결과 저장 및 출력
# ------------------------------------------------------------
for filename, content in [
    ("detector_label_map.json", detector_label_map),
    ("detector_data_summary.json", detector_data_summary),
]:
    (DETECTOR_DELIVERY_DIR / filename).write_text(
        json.dumps(content, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("========== 2. Dataset·DataLoader 생성 ==========")
print(f"선택 장치: {detector_check_device}")
print(f"배치 크기: {detector_check_config.batch_size}")
print(f"num_workers: {train_loader.num_workers}")
print(f"pin_memory: {train_loader.pin_memory}")
print(f"대회 이미지 준비: {len(required_codeit_images):,}장")
print("학습 증강: 사용하지 않음")
print("Validation 무작위 변환: 사용하지 않음")

print("\n[단일 클래스 매핑]")
print(json.dumps(detector_label_map, ensure_ascii=False, indent=2))

print("\n[전체 데이터 집계]")
print(json.dumps(detector_data_summary, ensure_ascii=False, indent=2))

print("\n[첫 배치 검사]")
for name, batch in [
    ("Train", train_check_batch),
    ("Validation", val_check_batch),
]:
    print(
        f"{name}: 이미지 {len(batch['images'])}장 / "
        f"첫 이미지 형태 {tuple(batch['images'][0].shape)} / "
        f"첫 이미지 라벨 {batch['targets'][0]['labels'].tolist()}"
    )

print("\n[통과] 전체 이미지 파일 존재 확인")
print("[통과] 전체 이미지·annotation ID 중복 및 연결 검사")
print("[통과] 분할 목록과 학습 라벨 일치")
print("[통과] Train·Validation 조합 그룹 중복 없음")
print("[통과] 전체 bbox 수치 및 경계 처리 후 양의 면적 확인")
print("[통과] 전체 annotation 개수 보존")
print("[통과] Train·Validation 첫 배치의 이미지·target·meta 형식")
print("[통과] 같은 설정으로 집계·첫 배치 재현")
print("[완료] 첫 배치 확인 후 로더 재생성")

if detector_data_summary["clipped_box_count"]:
    print(
        "\n[확인 필요] 경계 처리가 적용된 박스: "
        f"{detector_data_summary['clipped_box_count']:,}개"
    )

print("\n※ 전체 이미지의 픽셀 읽기를 반복한 검사는 아닙니다.")
print("※ 실제 알약과 bbox의 일치는 시각 검토가 필요합니다.")
print("※ 팀원의 검증 함수 연결과 모델 학습은 아직 수행하지 않았습니다.")
print("※ 기존 학습 라벨·분할·training_package.zip은 변경하지 않았습니다.")
print(f"\n새 코드와 검사 결과 저장 폴더:\n{DETECTOR_DELIVERY_DIR}")

## 3. 배치 시각화 및 탐지기 입력 검증

In [ ]:
# ============================================================
# 3. 배치 시각화 및 탐지기 입력 검증
# ============================================================

import importlib
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


# ------------------------------------------------------------
# 1. 출처별 표본 시각화
# ------------------------------------------------------------
review_samples = []

for split_name, loader in [
    ("train", train_loader),
    ("validation", val_loader),
]:
    dataset = loader.dataset

    for source_name in ["select", "codeit"]:
        sample_index = next(
            (
                index
                for index, record in enumerate(dataset.records)
                if record["dataset"] == source_name
            ),
            None,
        )

        if sample_index is None:
            print(
                f"[표본 없음] {split_name} / {source_name}"
            )
            continue

        sample = dataset[sample_index]

        # 시각화할 표본도 동일한 배치 검증 적용
        sample_batch = detector_data_module.collate_detector_samples(
            [sample]
        )
        detector_data_module.validate_batch(sample_batch)

        review_samples.append((split_name, source_name, sample))

if not review_samples:
    raise ValueError("시각화할 표본이 없습니다.")

fig, axes = plt.subplots(
    len(review_samples),
    1,
    figsize=(9, 8 * len(review_samples)),
    squeeze=False,
)

for row_index, (split_name, source_name, sample) in enumerate(
    review_samples
):
    image = sample["image"].permute(1, 2, 0).numpy()
    target = sample["target"]
    meta = sample["meta"]

    ax = axes[row_index, 0]
    ax.imshow(image)

    for box_index, box in enumerate(target["boxes"].tolist()):
        x1, y1, x2, y2 = box

        ax.add_patch(Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="red",
            linewidth=2,
        ))

        ax.text(
            x1,
            y1,
            (
                f"pill | ann {meta['annotation_ids'][box_index]}\n"
                f"{meta['drug_codes'][box_index]}"
            ),
            fontsize=9,
            color="black",
            va="bottom",
            bbox={
                "facecolor": "white",
                "alpha": 0.85,
                "edgecolor": "none",
            },
        )

    ax.set_title(
        f"{split_name} / {source_name} | "
        f"image {meta['image_id']} | "
        f"{len(target['boxes'])} boxes",
        fontsize=12,
    )
    ax.axis("off")

    print(
        f"\n[{split_name} / {source_name}] "
        f"이미지 ID: {meta['image_id']}"
    )
    print(f"원본 파일명: {meta['file_name']}")
    print(f"조합 그룹: {meta['group_id']}")
    print(f"박스 수: {meta['source_annotation_count']}")
    print(f"탐지 라벨: {target['labels'].tolist()}")

fig.tight_layout()

review_figure_path = (
    DETECTOR_DELIVERY_DIR / "detector_sample_review.png"
)
fig.savefig(review_figure_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\n시각화 저장: {review_figure_path}")
print(
    "※ 약품 코드는 추적용 표시이며, "
    "탐지 정답은 모든 알약에 대해 1입니다."
)


# ------------------------------------------------------------
# 2. 팀원의 실제 검증 함수 연결
# ------------------------------------------------------------
try:
    detector_api = importlib.import_module("detector")
except ModuleNotFoundError as error:
    if error.name != "detector":
        # detector 내부에서 다른 의존성이 누락된 경우 그대로 표시
        raise

    detector_api = None
    print(
        "\n[연결 대기] 현재 노트북에서 detector 모듈을 찾지 못했습니다."
    )
    print(
        "데이터 자체의 오류를 뜻하지 않습니다. "
        "팀원의 탐지기 코드가 있는 프로젝트에서 연결해야 합니다."
    )

if detector_api is not None:
    required_names = [
        "DetectorConfig",
        "validate_detection_batch",
        "validate_detector_data_summary",
        "validate_detector_label_map",
    ]

    missing_names = [
        name
        for name in required_names
        if not hasattr(detector_api, name)
    ]

    if missing_names:
        raise AttributeError(
            "detector 모듈에서 필요한 항목을 찾지 못했습니다: "
            + ", ".join(missing_names)
        )

    # 전달받은 실제 DetectorConfig 사용
    detector_config = detector_api.DetectorConfig(
        run_name="ssdlite320_baseline",
        output_dir=DETECTOR_DELIVERY_DIR / "outputs",
        label_map_path=(
            DETECTOR_DELIVERY_DIR / "detector_label_map.json"
        ),
        data_summary_path=(
            DETECTOR_DELIVERY_DIR / "detector_data_summary.json"
        ),
        learning_rate=5e-3,
        weight_decay=5e-4,
        score_threshold=0.25,
        nms_threshold=0.45,
        early_stopping_patience=7,
        seed=42,
        batch_size=8,
    )

    (
        train_loader,
        val_loader,
        detector_label_map,
        detector_data_summary,
    ) = detector_data_module.build_detector_inputs(
        config=detector_config,
        device=detector_check_device,
    )

    detector_api.validate_detector_label_map(
        detector_label_map
    )
    detector_api.validate_detector_data_summary(
        detector_data_summary,
        detector_label_map,
    )
    detector_api.validate_detection_batch(
        next(iter(train_loader)),
        detector_label_map,
    )
    detector_api.validate_detection_batch(
        next(iter(val_loader)),
        detector_label_map,
    )

    # 검증 후 학습에 사용할 로더를 다시 생성
    (
        train_loader,
        val_loader,
        detector_label_map,
        detector_data_summary,
    ) = detector_data_module.build_detector_inputs(
        config=detector_config,
        device=detector_check_device,
    )

    print("\n[통과] 팀원의 실제 DetectorConfig 연결")
    print("[통과] 팀원의 클래스 매핑 검증")
    print("[통과] 팀원의 데이터 집계 검증")
    print("[통과] 팀원의 Train·Validation 배치 검증")
    print("[완료] 검증 후 학습용 로더 재생성")

print("\n※ 모델 생성·학습은 실행하지 않았습니다.")
print("※ 원본 이미지·라벨·분할 목록은 변경하지 않았습니다.")

### 3-1. 실제 입력 검증 함수 실행

In [ ]:
# ============================================================
# 3. 팀원의 실제 입력 검증 함수 실행
# ============================================================

from pathlib import Path
from types import ModuleType
import hashlib
import json
import sys
import zipfile

# 전달받은 input_validation.py의 내용 식별값
expected_validation_hash = (
    "d4d31b49a333681c45d59638afa804e37"
    "c13beb273e573b9552c4e3fb901ba9e"
)

# 1. 현재 저장소와 알려진 프로젝트 위치에서 파일 탐색
search_roots = [
    Path.cwd(),
    *list(Path.cwd().parents)[:3],
    Path(DETECTOR_DATA_ROOT).parent,
    Path(DETECTOR_DATA_ROOT).parent / "gitkk-pill-detection",
    Path.home() / "Downloads",
    Path.home() / "Desktop",
]

validation_candidates = []
checked_paths = set()

for root in search_roots:
    for relative in [
        "detector/input_validation.py",
        "src/detector/input_validation.py",
        "gitkk-pill-detection/detector/input_validation.py",
        "gitkk-pill-detection/src/detector/input_validation.py",
    ]:
        path = root / relative

        if path.is_file() and str(path.resolve()) not in checked_paths:
            checked_paths.add(str(path.resolve()))
            validation_candidates.append((
                str(path.resolve()),
                path.read_bytes(),
            ))

    # 사용자가 올린 ZIP이 다운로드 폴더 등에 있는 경우도 지원
    zip_path = root / "detector.zip"

    if zip_path.is_file() and str(zip_path.resolve()) not in checked_paths:
        checked_paths.add(str(zip_path.resolve()))

        with zipfile.ZipFile(zip_path) as archive:
            members = [
                name
                for name in archive.namelist()
                if (
                    name.endswith("detector/input_validation.py")
                    and "__MACOSX" not in name
                )
            ]

            for member in members:
                validation_candidates.append((
                    f"{zip_path.resolve()} 내부 {member}",
                    archive.read(member),
                ))

matching_candidates = [
    (location, content)
    for location, content in validation_candidates
    if hashlib.sha256(content).hexdigest() == expected_validation_hash
]

if not matching_candidates:
    found_locations = "\n".join(
        location for location, _ in validation_candidates
    ) or "검증 파일을 찾지 못했습니다."

    raise FileNotFoundError(
        "전달받은 버전과 동일한 input_validation.py를 찾지 못했습니다.\n"
        "detector.zip을 다운로드 폴더에 두고 다시 실행하거나, "
        "현재 저장소의 detector 폴더 위치를 확인해 주세요.\n\n"
        f"탐색 결과:\n{found_locations}"
    )

validation_location, validation_content = matching_candidates[0]

# 2. 검증 파일만 독립적으로 불러오기
# detector/__init__.py와 모델 생성 코드는 실행하지 않습니다.
validation_module_name = "team_detector_input_validation"
team_validation = ModuleType(validation_module_name)
team_validation.__file__ = validation_location
sys.modules[validation_module_name] = team_validation

exec(
    compile(
        validation_content.decode("utf-8-sig"),
        validation_location,
        "exec",
    ),
    team_validation.__dict__,
)

# 3. 현재 설정으로 DataLoader 재생성
(
    train_loader,
    val_loader,
    detector_label_map,
    detector_data_summary,
) = detector_data_module.build_detector_inputs(
    config=detector_check_config,
    device=detector_check_device,
)

# 4. 팀원의 실제 검증 함수 실행
team_validation.validate_detector_label_map(
    detector_label_map
)

team_validation.validate_detector_data_summary(
    detector_data_summary,
    detector_label_map,
)

team_train_batch = next(iter(train_loader))
team_val_batch = next(iter(val_loader))

team_validation.validate_detection_batch(
    team_train_batch,
    detector_label_map,
)

team_validation.validate_detection_batch(
    team_val_batch,
    detector_label_map,
)

# 5. 첫 배치 확인 후 로더 재생성
(
    train_loader,
    val_loader,
    repeated_label_map,
    repeated_summary,
) = detector_data_module.build_detector_inputs(
    config=detector_check_config,
    device=detector_check_device,
)

assert repeated_label_map == detector_label_map
assert repeated_summary == detector_data_summary

# 6. 검증 근거 저장
team_validation_report = {
    "validation_file_sha256": expected_validation_hash,
    "split_hash": detector_data_summary["split_hash"],
    "label_map_validation": "passed",
    "data_summary_validation": "passed",
    "train_first_batch_validation": "passed",
    "validation_first_batch_validation": "passed",
    "num_train_images": detector_data_summary["num_train_images"],
    "num_val_images": detector_data_summary["num_val_images"],
    "num_train_instances": detector_data_summary["num_train_instances"],
    "num_val_instances": detector_data_summary["num_val_instances"],
    "full_detector_import_tested": False,
    "model_training_tested": False,
}

report_path = (
    Path(DETECTOR_DELIVERY_DIR) / "team_input_validation_report.json"
)

report_path.write_text(
    json.dumps(team_validation_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("========== 3-1 팀원의 실제 입력 검증 ==========")
print(f"검증 코드 위치: {validation_location}")
print("[통과] 전달받은 검증 파일과 내용 일치")
print("[통과] 단일 클래스 매핑")
print("[통과] 전체 데이터 집계")
print(
    f"[통과] Train 첫 배치: "
    f"{len(team_train_batch['images'])}장"
)
print(
    f"[통과] Validation 첫 배치: "
    f"{len(team_val_batch['images'])}장"
)
print("[완료] 검증 후 학습용 로더 재생성")
print(f"\n검증 기록 저장: {report_path}")
print("\n※ 팀원의 검증 파일을 수정 없이 실행했습니다.")
print("※ detector 전체 불러오기와 모델 학습은 아직 검사하지 않았습니다.")
print("※ 원본 이미지·학습 라벨·분할 목록은 변경하지 않았습니다.")

## 4. 팀원 공유용 패키지 생성

In [ ]:
# ============================================================
# 4. 팀원 공유용 패키지 생성
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import shutil
import zipfile

source_package = Path(DETECTOR_PACKAGE_DIR)
delivery_dir = Path(DETECTOR_DELIVERY_DIR)

share_version = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
share_result_dir = delivery_dir / f"share_{share_version}"
shared_package = share_result_dir / "training_package"
shared_package.mkdir(parents=True, exist_ok=False)


# ------------------------------------------------------------
# 1. 입력 검증 완료 기록 확인
# ------------------------------------------------------------
with (
    delivery_dir / "team_input_validation_report.json"
).open("r", encoding="utf-8") as file:
    validation_report = json.load(file)

with (
    delivery_dir / "detector_data_summary.json"
).open("r", encoding="utf-8") as file:
    reference_summary = json.load(file)

for field in [
    "label_map_validation",
    "data_summary_validation",
    "train_first_batch_validation",
    "validation_first_batch_validation",
]:
    if validation_report.get(field) != "passed":
        raise ValueError(f"입력 검증 완료 기록 확인 필요: {field}")

if validation_report["split_hash"] != reference_summary["split_hash"]:
    raise ValueError("검증 기록과 데이터 집계의 분할 해시가 다릅니다.")


# ------------------------------------------------------------
# 2. 최종 라벨과 분할 자료 복사
# ------------------------------------------------------------
expected_counts = {
    "train": (8256, 31673),
    "validation": (2063, 7915),
}

for split_name, (image_count, box_count) in expected_counts.items():
    source = source_package / "annotations" / f"{split_name}.json"

    with source.open("r", encoding="utf-8") as file:
        document = json.load(file)

    if len(document["images"]) != image_count:
        raise ValueError(f"{split_name}: 최종 이미지 수 불일치")

    if len(document["annotations"]) != box_count:
        raise ValueError(f"{split_name}: 최종 박스 수 불일치")

    destination = (
        shared_package / "annotations" / f"{split_name}.json"
    )
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

for filename in [
    "split_manifest.csv",
    "train_manifest.csv",
    "validation_manifest.csv",
    "class_mapping.csv",
]:
    source = source_package / filename

    if not source.is_file():
        raise FileNotFoundError(f"필수 파일 없음: {source}")

    shutil.copy2(source, shared_package / filename)

shared_module_dir = shared_package / "data"
shared_module_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    delivery_dir / "data" / "detector_data.py",
    shared_module_dir / "detector_data.py",
)

(shared_module_dir / "__init__.py").write_text(
    "", encoding="utf-8"
)

for filename in [
    "detector_label_map.json",
    "detector_data_summary.json",
    "team_input_validation_report.json",
]:
    shutil.copy2(
        delivery_dir / filename,
        shared_package / filename,
    )


# ------------------------------------------------------------
# 3. 팀원 노트북의 첫 번째 셀: 경로 설정
# ------------------------------------------------------------
setup_code = r'''
from pathlib import Path
import importlib
import json
import shutil
import sys

# 본인 컴퓨터에 맞게 아래 두 경로를 수정합니다.
PROJECT_ROOT = Path("/본인/탐지기/저장소")
SHARED_ROOT = Path("/본인/공유자료/폴더")

# 공유자료 폴더 내부 구조
# select/images/
# codeit_train_validated/images/
# training_package/

PROJECT_ROOT = PROJECT_ROOT.expanduser().resolve()
SHARED_ROOT = SHARED_ROOT.expanduser().resolve()

PACKAGE_DIR = SHARED_ROOT / "training_package"
SELECT_ROOT = SHARED_ROOT / "select"
CODEIT_ROOT = SHARED_ROOT / "codeit_train_validated"

required_paths = [
    PROJECT_ROOT / "detector" / "__init__.py",
    PACKAGE_DIR / "annotations" / "train.json",
    PACKAGE_DIR / "annotations" / "validation.json",
    PACKAGE_DIR / "split_manifest.csv",
    PACKAGE_DIR / "data" / "detector_data.py",
    SELECT_ROOT / "images",
    CODEIT_ROOT / "images",
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"필요한 파일 또는 폴더가 없습니다: {path}"
        )

project_data_dir = PROJECT_ROOT / "data"

# 다른 위치의 동명 모듈이 이미 로드된 경우 확인
for module_name, expected_dir in [
    ("data", project_data_dir),
    ("detector", PROJECT_ROOT / "detector"),
]:
    loaded = sys.modules.get(module_name)

    if loaded is not None:
        loaded_file = getattr(loaded, "__file__", None)

        if (
            loaded_file is None
            or Path(loaded_file).resolve().parent
            != expected_dir.resolve()
        ):
            raise RuntimeError(
                f"다른 위치의 {module_name} 모듈이 로드되어 있습니다. "
                "커널을 다시 시작하고 처음부터 실행하세요."
            )

# 탐지기 저장소에 데이터 모듈 설치
project_data_dir.mkdir(parents=True, exist_ok=True)

source_module = PACKAGE_DIR / "data" / "detector_data.py"
target_module = project_data_dir / "detector_data.py"

if target_module.is_file():
    if target_module.read_bytes() != source_module.read_bytes():
        shutil.copy2(
            target_module,
            target_module.with_name("detector_data.py.previous"),
        )

shutil.copy2(source_module, target_module)

init_path = project_data_dir / "__init__.py"
if not init_path.exists():
    init_path.write_text("", encoding="utf-8")

# 각 컴퓨터의 실제 경로를 별도 설정 파일에 저장
settings = {
    "package_dir": str(PACKAGE_DIR),
    "image_roots": {
        "select": str(SELECT_ROOT),
        "codeit": str(CODEIT_ROOT),
    },
    "augmentation": {
        "horizontal_flip": 0.0,
        "vertical_flip": 0.0,
        "brightness": 0.0,
        "contrast": 0.0,
    },
}

(project_data_dir / "detector_data_settings.json").write_text(
    json.dumps(settings, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

project_path = str(PROJECT_ROOT)
if project_path in sys.path:
    sys.path.remove(project_path)

sys.path.insert(0, project_path)
importlib.invalidate_caches()

print("데이터 경로와 모듈 설치 완료")
print(f"탐지기 저장소: {PROJECT_ROOT}")
print(f"학습 패키지: {PACKAGE_DIR}")
'''


# ------------------------------------------------------------
# 4. 팀원 노트북의 두 번째 셀: 실제 입력 검증
# ------------------------------------------------------------
validation_code = r'''
import hashlib
import importlib
import json

from detector import (
    DetectorConfig,
    select_device,
    validate_detection_batch,
    validate_detector_data_summary,
    validate_detector_label_map,
)

detector_data = importlib.import_module("data.detector_data")
detector_data = importlib.reload(detector_data)

expected_module_path = (
    PROJECT_ROOT / "data" / "detector_data.py"
).resolve()

if Path(detector_data.__file__).resolve() != expected_module_path:
    raise RuntimeError("설정한 데이터 구성 모듈을 불러오지 못했습니다.")

validation_module = importlib.import_module(
    "detector.input_validation"
)

with (
    PACKAGE_DIR / "team_input_validation_report.json"
).open("r", encoding="utf-8") as file:
    validation_reference = json.load(file)

current_validation_hash = hashlib.sha256(
    Path(validation_module.__file__).read_bytes()
).hexdigest()

if (
    current_validation_hash
    != validation_reference["validation_file_sha256"]
):
    raise RuntimeError(
        "공유 당시와 탐지기 검증 코드의 버전이 다릅니다. "
        "팀원 간 코드 버전을 먼저 맞춰 주세요."
    )

config = DetectorConfig(
    run_name="ssdlite320_baseline",
    output_dir=PROJECT_ROOT / "outputs",
    label_map_path=(
        PROJECT_ROOT / "data" / "detector_label_map.json"
    ),
    data_summary_path=(
        PROJECT_ROOT / "data" / "detector_data_summary.json"
    ),
    learning_rate=5e-3,
    weight_decay=5e-4,
    score_threshold=0.25,
    nms_threshold=0.45,
    early_stopping_patience=7,
    seed=42,
    batch_size=8,
)

device = select_device()
data_factory_path = "data.detector_data:build_detector_inputs"

module_name, function_name = data_factory_path.split(":", 1)
data_factory = getattr(
    importlib.import_module(module_name),
    function_name,
)

train_loader, val_loader, label_map, data_summary = data_factory(
    config=config,
    device=device,
)

validate_detector_label_map(label_map)
validate_detector_data_summary(data_summary, label_map)

for loader in [train_loader, val_loader]:
    batch = next(iter(loader))
    validate_detection_batch(batch, label_map)

with (
    PACKAGE_DIR / "detector_data_summary.json"
).open("r", encoding="utf-8") as file:
    reference_summary = json.load(file)

if data_summary != reference_summary:
    raise ValueError("공유 기준과 현재 데이터 집계가 다릅니다.")

with (
    PACKAGE_DIR / "detector_label_map.json"
).open("r", encoding="utf-8") as file:
    reference_label_map = json.load(file)

if label_map != reference_label_map:
    raise ValueError("공유 기준과 현재 클래스 매핑이 다릅니다.")

for path, content in [
    (config.label_map_path, label_map),
    (config.data_summary_path, data_summary),
]:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(
        json.dumps(content, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

# 첫 배치 검사 후 학습에 사용할 로더를 다시 생성
train_loader, val_loader, label_map, data_summary = data_factory(
    config=config,
    device=device,
)

print("탐지기 입력 검증 완료")
print(f"장치: {device}")
print(f"배치 크기: {config.batch_size}")
print(f"Train: {data_summary['num_train_images']:,}장")
print(f"Validation: {data_summary['num_val_images']:,}장")
print(f"분할 해시: {data_summary['split_hash']}")
print("모델 생성과 학습은 탐지기 학습 코드에서 진행하세요.")
'''


# ------------------------------------------------------------
# 5. 코드 문법 확인 및 실행 노트북 저장
# ------------------------------------------------------------
module_source = (
    shared_module_dir / "detector_data.py"
).read_text(encoding="utf-8")

for name, code in [
    ("경로 설정", setup_code),
    ("입력 검증", validation_code),
    ("데이터 구성 모듈", module_source),
]:
    compile(code, name, "exec")


def make_code_cell(source, cell_id):
    return {
        "cell_type": "code",
        "id": cell_id,
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": source.splitlines(keepends=True),
    }


notebook = {
    "cells": [
        {
            "cell_type": "markdown",
            "id": "instructions",
            "metadata": {},
            "source": [
                "# 단일 클래스 알약 탐지 데이터 연결\n",
                "\n",
                "첫 코드 셀에서 PROJECT_ROOT와 SHARED_ROOT를 수정합니다.\n",
                "팀원의 탐지기 코드와 해당 실행 환경이 필요합니다.\n",
                "셀을 위에서부터 순서대로 실행합니다.\n",
                "이 노트북은 입력 검증까지 수행하며 학습을 시작하지 않습니다.\n",
            ],
        },
        make_code_cell(setup_code, "setup"),
        make_code_cell(validation_code, "validation"),
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        },
        "language_info": {
            "name": "python",
        },
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

(shared_package / "START_HERE.ipynb").write_text(
    json.dumps(notebook, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. 팀원용 안내서 저장
# ------------------------------------------------------------
readme = """# 단일 클래스 알약 탐지 데이터 사용 안내

## 1. 최종 데이터

- Train: 이미지 8,256장 / 박스 31,673개
- Validation: 이미지 2,063장 / 박스 7,915개
- Train 조합 그룹: 2,870개
- Validation 조합 그룹: 717개
- 분할 간 조합 그룹 중복: 0개

기존 공통 분할을 사용하며 각자 다시 분할하지 않습니다.

## 2. 탐지 클래스

- background: 0
- pill: 1
- 모델 클래스 수: background를 포함해 2
- 모든 알약의 탐지 target label은 1

약품 코드 118종은 원본 추적과 분류기 연결을 위해 보존합니다.
class_mapping.csv의 약품별 번호를 탐지 target으로 사용하지 않습니다.

## 3. 준비할 폴더

공유자료 폴더 안에 다음 세 폴더를 배치합니다.

- select/images/
- codeit_train_validated/images/
- training_package/

SELECT 이미지는 기존 공유 자료를 그대로 사용합니다.
대회 이미지는 기존 codeit_train_validated.zip을 풀어 준비합니다.
이미지 경로가 중첩되지 않았는지 확인합니다.

training_package는 이번 ZIP을 풀어서 준비합니다.
이전 버전의 학습 패키지와 파일을 섞지 않습니다.

이번 ZIP에는 이미지가 포함되어 있지 않습니다.
학습에 사용할 이미지는 최종 라벨과 분할 목록으로 결정합니다.

## 4. 실행 환경

팀원의 detector 폴더가 있는 탐지기 저장소가 필요합니다.
해당 저장소의 설치 안내에 맞춘 환경을 사용합니다.

데이터 모듈은 torch, numpy, Pillow를 사용합니다.
detector 전체를 불러오려면 torchvision, torchmetrics 등
탐지기 코드에서 사용하는 의존성도 필요합니다.

## 5. 실행 순서

1. START_HERE.ipynb를 엽니다.
2. PROJECT_ROOT에 detector 폴더의 상위 저장소 경로를 입력합니다.
3. SHARED_ROOT에 위 세 폴더가 들어 있는 경로를 입력합니다.
4. 첫 셀을 실행하여 데이터 모듈과 경로 설정을 준비합니다.
5. 두 번째 셀을 실행하여 실제 입력 검증을 수행합니다.
6. 탐지기 입력 검증 완료 메시지를 확인합니다.

첫 셀은 저장소의 data/detector_data.py를 설치합니다.
기존 파일과 내용이 다르면 .previous 사본을 보관합니다.
기존 data/__init__.py는 덮어쓰지 않습니다.

## 6. 학습 Notebook 연결

사용할 데이터 구성 함수 경로:

data_factory_path = "data.detector_data:build_detector_inputs"

학습 Notebook에서 같은 함수 경로와 DetectorConfig를 사용합니다.
첫 배치 검증 후에는 DataLoader를 다시 생성합니다.

모델 생성과 학습 호출은 팀원의 탐지기 코드가 담당합니다.
START_HERE.ipynb 자체는 학습을 시작하지 않습니다.

## 7. 기본 DataLoader 설정

- seed: 42
- batch_size: 8
- CUDA: num_workers=2, pin_memory=True
- MPS 또는 CPU: num_workers=0, pin_memory=False
- Train만 셔플
- 마지막 작은 배치도 유지
- CPU의 RGB float32 이미지
- 이미지 값 범위: 0~1
- images, targets, meta는 각각 리스트
- 외부 resize 및 ImageNet 정규화 없음
- bbox 좌표 형식: XYXY
- 면적: 최종 bbox에서 재계산
- 모든 target label: 1
- 이번 기준 실행에서는 증강을 사용하지 않음

학습률과 임계값의 예시는 전달받은 팀원 설정을 따릅니다.
다른 모델 담당자는 해당 모델 입력 명세와의 호환성을 확인합니다.

## 8. 주요 파일

- annotations/train.json: 최종 Train 라벨
- annotations/validation.json: 최종 Validation 라벨
- split_manifest.csv: 전체 이미지 공통 분할
- train_manifest.csv: Train 이미지 목록
- validation_manifest.csv: Validation 이미지 목록
- class_mapping.csv: 원본 약품별 클래스 번호
- detector_label_map.json: 단일 클래스 탐지 번호
- detector_data_summary.json: 데이터 집계와 분할 해시
- team_input_validation_report.json: 실제 입력 검증 완료 기록
- data/detector_data.py: Dataset과 DataLoader 구성
- START_HERE.ipynb: 경로 설정 및 입력 검증
- package_checksums.json: 공유 파일별 내용 식별값

## 9. 확인한 범위

- 최종 이미지 및 박스 집계 일치
- 전체 이미지 파일 경로 존재
- 이미지와 annotation ID 중복 없음
- 라벨과 분할 목록 연결
- 분할 간 조합 그룹 중복 없음
- 전체 bbox 수치 및 양의 면적 확인
- 팀원의 실제 클래스 매핑 검증 통과
- 팀원의 실제 데이터 집계 검증 통과
- 팀원의 실제 Train 첫 배치 검증 통과
- 팀원의 실제 Validation 첫 배치 검증 통과
- 같은 설정에서 집계와 첫 배치 재현
- 분할별 SELECT 및 대회 표본 총 4장 시각 검토 통과

팀원 환경에서의 detector 전체 불러오기와 모델 학습은
각 실행 환경에서 이어서 확인합니다.

전체 이미지의 실제 알약과 박스가 일치하는지 자동으로
판정한 검사는 아닙니다.
"""

(shared_package / "README.md").write_text(
    readme, encoding="utf-8"
)


# ------------------------------------------------------------
# 7. 파일 식별값 기록 및 ZIP 생성
# ------------------------------------------------------------
def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


file_hashes = {
    path.relative_to(shared_package).as_posix(): file_sha256(path)
    for path in sorted(shared_package.rglob("*"))
    if path.is_file()
}

(shared_package / "package_checksums.json").write_text(
    json.dumps(file_hashes, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

shared_zip = share_result_dir / "training_package_detector.zip"

with zipfile.ZipFile(
    shared_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(shared_package.rglob("*")):
        if path.is_file():
            relative = path.relative_to(shared_package)

            archive.write(
                path,
                arcname=(
                    Path("training_package") / relative
                ).as_posix(),
            )


# ------------------------------------------------------------
# 8. ZIP 저장 결과 확인
# ------------------------------------------------------------
with zipfile.ZipFile(shared_zip) as archive:
    damaged_member = archive.testzip()

    if damaged_member is not None:
        raise ValueError(f"ZIP 무결성 검사 실패: {damaged_member}")

    for relative, expected_hash in file_hashes.items():
        content = archive.read(f"training_package/{relative}")
        actual_hash = hashlib.sha256(content).hexdigest()

        if actual_hash != expected_hash:
            raise ValueError(f"ZIP 내부 파일 내용 불일치: {relative}")

print("========== 4. 팀원 공유용 패키지 생성 ==========")
print("[통과] 팀원의 입력 검증 완료 기록 확인")
print("[통과] 최종 이미지 및 박스 수 확인")
print("[통과] 실행 노트북과 데이터 모듈 문법 검사")
print("[통과] 공유 ZIP 무결성 및 파일 내용 비교")
print("[완료] 라벨·DataLoader·검증 기록·실행 노트북·안내서 저장")

print(f"\n공유 ZIP:\n{shared_zip}")
print(f"\n압축 전 폴더:\n{shared_package}")
print(f"\nZIP 크기: {shared_zip.stat().st_size / 1024**2:.1f} MB")
print(f"ZIP SHA-256: {file_sha256(shared_zip)}")

print("\n※ 이미지 파일은 ZIP에 포함하지 않았습니다.")
print("※ 기존 학습 패키지와 원본 데이터는 변경하지 않았습니다.")
print("※ 생성한 실행 노트북은 팀원 환경에서 실행 확인이 필요합니다.")
print("※ 모델 생성 및 학습은 아직 수행하지 않았습니다.")